##  AgERA5 Weather Extraction per NUTS3

Note: the AgERA5 service is only reachable on the WUR network.


%pip install sqlitedict

In [1]:
import pandas as pd
import sqlite3, pickle, zlib
from sqlitedict import SqliteDict
from types import SimpleNamespace
from pathlib import Path
import config
from weather_provider import WOFOSTWebWeatherDataProvider

In [2]:
# Caching Function
def get_weatherdata(ccnl_field, field_info, start_date, end_date):
    """Retrieve and cache weather data from AgERA5.
    """
 
    def my_encode(obj):
        return sqlite3.Binary(zlib.compress(pickle.dumps(obj, pickle.HIGHEST_PROTOCOL)))
 
    def my_decode(obj):
        return pickle.loads(zlib.decompress(bytes(obj)))
 
    with SqliteDict(config.meteo_cache, encode=my_encode, decode=my_decode, autocommit=True) as db:
        if ccnl_field.fieldid in db:
            wdp = db[ccnl_field.fieldid]
        else:
            host = config.weather_host
            inputs = dict(hostname=host, port=8080, latitude=field_info.latitude,
                          longitude=field_info.longitude, startdate=start_date, enddate=end_date)
            wdp = WOFOSTWebWeatherDataProvider(inputs)
            db[ccnl_field.fieldid] = wdp
 
    return wdp

In [ ]:
# Read selected marginal NUTS3 regions with centroid coordinates (see notebook nuts3_centroid.ipynb)
folder = Path.cwd().parent
sites = pd.read_csv(folder / "sites_geopandas" / "nuts3_centroid.csv")
START, END = "20181001", "20231231"
sites.head()

# Build stand-ins for defined function
wdps ={}
for s in sites.itertuples():
    ccnl_field = SimpleNamespace(fieldid=s.nuts3)
    field_info = SimpleNamespace(latitude=s.lat, longitude=s.lon)
    try:
        wdps[s.nuts3] = get_weatherdata(ccnl_field, field_info, START, END)
        print(f"{s.nuts3:6s} OK")
    except Exception as e:
        print(f"{s.nuts3:6s} FAILED -- {e}")

print(f"\n{len(wdps)}/{len(sites)} sites cached.")

AL111  OK
AT111  OK
AT121  OK
AT124  OK
AT211  OK
AT212  OK
AT213  OK
AT222  OK
AT223  OK
AT226  OK
AT314  OK
AT315  OK
AT321  OK
AT322  OK
AT323  OK
AT331  OK
AT332  OK
AT333  OK
AT334  OK
AT335  OK
AT341  OK
AT342  OK
BA111  OK
BE336  OK
BG411  OK
BG413  OK
BG423  OK
BG424  OK
CZ041  OK
DE132  OK
DE136  OK
DE137  OK
DE139  OK
DE141  OK
DE143  OK
DE147  OK
DE148  OK
DE213  OK
DE215  OK
DE216  OK
DE218  OK
DE21D  OK
DE21F  OK
DE21K  OK
DE21L  OK
DE21M  OK
DE21N  OK
DE225  OK
DE249  OK
DE24D  OK
DE263  OK
DE272  OK
DE273  OK
DE27A  OK
DE27B  OK
DE27E  OK
DE401  OK
DE405  OK
DE406  OK
DE407  OK
DE408  OK
DE40A  OK
DE40B  OK
DE40C  OK
DE40D  OK
DE40E  OK
DE40F  OK
DE40G  OK
DE40H  OK
DE803  OK
DE80L  OK
DE80M  OK
DE937  OK
DE948  OK
DE949  OK
DE94B  OK
DEA2D  OK
DEA34  OK
DEA37  OK
DEA57  OK
DEA59  OK
DEA5A  OK
DEB3A  OK
DEC05  OK
DED42  OK
DEE06  OK
DEE0D  OK
DEE0E  OK
DEG03  OK
DEG04  OK
DK011  OK
DK014  OK
DK032  OK
DK041  OK
EE001  OK
EE004  OK
EE006  OK
EE007  OK
EE008  OK
EL302  OK


In [4]:
# unit sanity check: confirm the provider already returns PCSE internal units
import datetime as dt

w = wdps[sites.iloc[0]["nuts3"]]
print(w)                          # summary: coordinates + available date range
print(w(dt.date(2020, 7, 1)))     # one mid-summer day (pick a date inside your window)

# Expected (no conversion needed):
#   IRRAD  J/m2/day  (~1e7)   |  VAP  hPa (~5-30)  |  RAIN  cm/day
#   TMIN / TMAX / TEMP  Celsius  |  WIND  m/s  |  SNOWDEPTH  cm

Weather data provided by: WOFOSTWebWeatherDataProvider
--------Description---------
----Site characteristics----
Elevation:  377.0
Latitude:  41.100
Longitude: 20.100
Data available for 2018-10-01 - 2023-12-31
Number of missing days: 0

Weather data for 2020-07-01 (DAY)
IRRAD:  26191354.00  J/m2/day
 TMIN:        17.30   Celsius
 TMAX:        31.38   Celsius
  VAP:        20.71       hPa
 RAIN:         0.00    cm/day
   E0:         0.62    cm/day
  ES0:         0.54    cm/day
  ET0:         0.52    cm/day
 WIND:         0.64     m/sec
SNOWDEPTH:         0.00        cm
 TEMP:        25.35   Celsius
Latitude  (LAT):    41.10 degr.
Longitude (LON):    20.10 degr.
Elevation (ELEV):  377.0 m.

